In [1]:
import pandas as pd
pd.options.mode.string_storage = "python"
import mmml

In [2]:
import jax
jax.devices()

[CudaDevice(id=0), CudaDevice(id=1)]

In [3]:
import ase
from ase import io as ase_io
from ase.visualize import view

In [4]:
lala = ase.io.read("lala.xyz")

In [5]:
rala = ase.io.read("rala.xyz")

In [6]:
view(lala, viewer="x3d")

In [7]:
view(rala, viewer="x3d")

In [11]:

import chemcoord as cc
lala_cc = cc.Cartesian.read_xyz("lala.xyz")
lala_cc

,atom,x,y,z
0,H,-2.1243,-0.0920,1.6298
1,N,-1.7374,-0.1422,0.7112
2,H,-2.0613,0.6335,0.1718
3,C,-0.2575,-0.1205,0.8106
4,H,0.0277,-1.0425,1.3783
5,C,0.3039,1.1050,1.5253
6,H,-0.0733,1.1716,2.5550
7,H,0.0396,2.0418,1.0151
8,H,1.3991,1.0613,1.5830
9,C,0.3219,-0.1861,-0.5965


In [37]:
import numpy as np
import pandas as pd
import chemcoord as cc
import importlib


def patch_chemcoord_for_pandas3():
    gz = importlib.import_module(
        "chemcoord.cartesian_coordinates._cartesian_class_get_zmat"
    )
    zc = importlib.import_module(
        "chemcoord.internal_coordinates._zmat_class_core"
    )

    # Resolve internals from the module namespace first, instead of guessing paths
    transformation = getattr(zc, "transformation", None)
    constants = getattr(zc, "constants", None)
    replace_without_warn = getattr(zc, "replace_without_warn", None)

    # Fallbacks if the names are not already present in _zmat_class_core
    if constants is None:
        constants = importlib.import_module("chemcoord.constants")

    if replace_without_warn is None:
        common_methods = importlib.import_module("chemcoord.common_methods")
        replace_without_warn = getattr(common_methods, "replace_without_warn")

    if transformation is None:
        # Try a few likely private-module locations used across versions
        candidates = [
            "chemcoord.internal_coordinates._zmat_transformation",
            "chemcoord.internal_coordinates.zmat_transformation",
            "chemcoord._internal_coordinates._zmat_transformation",
            "chemcoord._internal_coordinates.zmat_transformation",
        ]
        for modname in candidates:
            try:
                transformation = importlib.import_module(modname)
                break
            except ModuleNotFoundError:
                pass

    if transformation is None:
        raise ModuleNotFoundError(
            "Could not locate chemcoord's internal zmat transformation module. "
            "Run `print(dir(importlib.import_module('chemcoord.internal_coordinates._zmat_class_core')))` "
            "to inspect available names."
        )

    # Resolve the Cartesian constructor without assuming exact file layout
    Cartesian_cls = getattr(cc, "Cartesian", None)
    if Cartesian_cls is None:
        cm = importlib.import_module(
            "chemcoord.cartesian_coordinates.cartesian_class_main"
        )
        Cartesian_cls = getattr(cm, "Cartesian")

    # --- Patch 1: make b/a/d object dtype, not strict string dtype ---
    def _build_zmat_patched(self, construction_table):
        c_table = construction_table.copy()

        for col in ["b", "a", "d"]:
            if col in c_table.columns:
                c_table[col] = c_table[col].astype(object)

        zmat_frame = pd.DataFrame(index=c_table.index)

        # atom labels should also avoid pandas string-arrow strictness here
        zmat_frame["atom"] = self.loc[c_table.index, "atom"].astype(object)

        for col in ["b", "a", "d"]:
            zmat_frame[col] = pd.Series(index=c_table.index, dtype=object)

        zmat_frame.loc[:, ["b", "a", "d"]] = (
            c_table.loc[:, ["b", "a", "d"]].astype(object)
        )

        zmat_values = np.asarray(
            self._calculate_zmat_values(c_table),
            dtype=np.float64,
        )
        zmat_frame["bond"] = zmat_values[:, 0]
        zmat_frame["angle"] = zmat_values[:, 1]
        zmat_frame["dihedral"] = zmat_values[:, 2]

        Zmat_cls = getattr(gz, "Zmat", None) or getattr(cc, "Zmat", None)
        if Zmat_cls is None:
            return zmat_frame
        return Zmat_cls(zmat_frame)

    gz.CartesianGetZmat._build_zmat = _build_zmat_patched

    # --- Patch 2: force writable numeric arrays in get_cartesian ---
    def get_cartesian_patched(self):
        c_table = self.loc[:, ["b", "a", "d"]].copy()

        c_table = (
            replace_without_warn(c_table, constants.int_label)
            .astype("i8")
            .replace({k: v for v, k in enumerate(c_table.index)})
            .to_numpy(copy=True)
            .T
        )

        C = self.loc[:, ["bond", "angle", "dihedral"]].to_numpy(
            dtype=np.float64,
            copy=True,
        ).T

        C[[1, 2], :] = np.radians(C[[1, 2], :])

        err, row, positions = transformation.get_X(C, c_table)
        positions = positions.T

        if err:
            raise ValueError(f"Error in row {row} with positions {positions}")

        cartesian = pd.DataFrame(
            positions,
            index=self.index,
            columns=["x", "y", "z"],
        )
        cartesian["atom"] = self["atom"].astype(object)

        return Cartesian_cls(cartesian)

    zc.ZmatCore.get_cartesian = get_cartesian_patched


patch_chemcoord_for_pandas3()


def wrap_deg(x):
    return (x + 180.0) % 360.0 - 180.0

def interpolate_zmats(z1, z2, steps):
    base = z1.copy()

    c1 = z1.loc[:, ["bond", "angle", "dihedral"]].copy()
    c2 = z2.loc[:, ["bond", "angle", "dihedral"]].copy()

    delta = c2 - c1
    delta["angle"] = wrap_deg(delta["angle"])
    delta["dihedral"] = wrap_deg(delta["dihedral"])

    out = []
    for i in range(steps + 1):
        t = i / steps
        z = base.copy()
        vals = c1 + t * delta

        # avoid zero-length bonds during interpolation
        vals["bond"] = vals["bond"].clip(lower=1e-4)

        z.unsafe_loc[:, ["bond", "angle", "dihedral"]] = vals
        out.append(z)

    return out





In [ ]:
def interpolate_xyzs_to_npz(xyz1: str, xyz2: str, out_fn="interpolated.npz"):

    lala_zm = cc.Cartesian.read_xyz(xyz1).get_zmat()
    rala_cc = cc.Cartesian.read_xyz(xyz2)
    c_table = lala_zm.loc[:, ["b", "a", "d"]]
    rala_zm = rala_cc.get_zmat(c_table)
    # interpolate between structures
    mixed = interpolate_zmats(lala_zm, rala_zm, 1000)
    mixed_ccs = [_.get_cartesian() for _ in  mixed]
    ase_atoms_list = [ase.Atoms(mixed_ccs[i]["atom"], mixed_ccs[i][["x", "y", "z"]]) for i in range(STEPS)]

    # write

In [42]:
lala_zm = cc.Cartesian.read_xyz("lala.xyz").get_zmat()
print(lala_zm)
rala_cc = cc.Cartesian.read_xyz("rala.xyz")
c_table = lala_zm.loc[:, ["b", "a", "d"]]
rala_zm = rala_cc.get_zmat(c_table)
# interpolate between structures
mixed = interpolate_zmats(lala_zm, rala_zm, 1000)
mixed_ccs = [_.get_cartesian() for _ in  mixed]

   atom       b    a    d      bond       angle    dihedral
3     C  origin  e_z  e_x  0.859010   19.327191 -154.922231
1     N       3  e_z  e_x  1.483393  142.590642  155.226943
5     C       3    1  e_x  1.525718  114.219323 -112.467504
9     C       3    1    5  1.523134  108.479797  121.925547
4     H       3    1    9  1.119690  106.024121  117.684854
0     H       1    3    9  0.998017  108.925821  179.547182
2     H       1    3    9  0.998786  110.386858   59.402250
8     H       5    3    1  1.097589  111.088052 -179.151567
6     H       5    3    1  1.098634  111.186338  -59.850211
7     H       5    3    1  1.098978  112.255117   60.631876
10    O       9    3    1  1.217113  129.078958  -54.396715
11    O       9    3    1  1.352468  114.864537  126.679985
12    H      11    9    3  0.952195  110.083083 -179.486271


In [48]:
i = 0
view(ase_atoms_list[i], viewer="x3d")

In [50]:
i = 500
view(ase_atoms_list[i], viewer="x3d")

In [49]:
i = -1
view(ase_atoms_list[i], viewer="x3d")